### Validate the cleaned telemetry data

Check row and column counts, missing values, duplicate records and business keys, and invalid values before building the warehouse.

In [25]:
#Data quality Checks
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType

dq_df = spark.table("processed_telemetry")

# 1. Record and column counts
print("Total records:", dq_df.count())
print("Total columns:", len(dq_df.columns))
print("Column names:", dq_df.columns)

# 2. NULL counts for all columns
null_exprs = []

for field in dq_df.schema.fields:
    c = field.name
    condition = F.col(c).isNull()

    if isinstance(field.dataType, (DoubleType, FloatType)):
        condition = condition | F.isnan(F.col(c))

    null_exprs.append(
        F.count(F.when(condition, 1)).alias(c)
    )

print("NULL counts:")
display(dq_df.agg(*null_exprs))

# 3. Fully duplicated rows
duplicate_rows = (
    dq_df.groupBy(*dq_df.columns)
         .count()
         .filter(F.col("count") > 1)
         .agg(
             F.coalesce(
                 F.sum(F.col("count") - 1), F.lit(0)
             ).alias("duplicate_rows")
         )
         .first()["duplicate_rows"]
)

print("Fully duplicated extra records:", duplicate_rows)

# 4. Duplicate business keys
key_cols = ["timestamp", "server_id", "experiment_id"]

duplicate_key_groups = (
    dq_df.groupBy(*key_cols)
         .count()
         .filter(F.col("count") > 1)
         .count()
)

print("Duplicate key groups:", duplicate_key_groups)

# 5. Invalid-value checks
invalid_conditions = {
    "GPU utilization outside 0-100":
        (F.col("gpu_utilization_avg_pct") < 0) |
        (F.col("gpu_utilization_avg_pct") > 100),

    "CPU utilization outside 0-100":
        (F.col("cpu_utilization_percent") < 0) |
        (F.col("cpu_utilization_percent") > 100),

    "Negative GPU power":
        F.col("gpu_power_total_W") < 0,

    "Negative CPU power":
        F.col("cpu_power_W") < 0,

    "Negative GPU memory used":
        F.col("gpu_memory_used_total_GB") < 0,

    "GPU memory used exceeds total":
        F.col("gpu_memory_used_total_GB") >
        F.col("gpu_memory_total_GB"),

    "Non-positive GPU count":
        F.col("gpu_count") <= 0
}

invalid_counts = [
    F.count(
        F.when(condition, 1)
    ).alias(name)
    for name, condition in invalid_conditions.items()
]

print("Invalid-value counts:")
display(dq_df.agg(*invalid_counts))

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 26, Finished, Available, Finished, False)

Total records: 2000000
Total columns: 27
Column names: ['timestamp', 'date', 'time', 'server_id', 'model_id', 'experiment_id', 'gpu_count', 'gpu_utilization_avg_pct', 'gpu_power_total_W', 'gpu_temp_max_C', 'cpu_utilization_percent', 'cpu_freq_MHz', 'cpu_power_W', 'cpu_temp_C', 'ambient_temp_C', 'inlet_temp_C', 'exhaust_temp_C', 'humidity_pct', 'cooling_type', 'cooling_power_W', 'fan_speed_RPM', 'airflow_CFM', 'coolant_temp_C', 'coolant_flow_rate', 'gpu_memory_used_total_GB', 'gpu_memory_total_GB', 'cpu_freq_GHz']
NULL counts:


SynapseWidget(Synapse.DataFrame, 3bb1b53b-34dd-48ad-a5e9-168af5240ad2)

Fully duplicated extra records: 0
Duplicate key groups: 0
Invalid-value counts:


SynapseWidget(Synapse.DataFrame, e7f8dffe-3dc8-4e78-8416-808ef816b60a)

### Inspect the server dimension source

Review the server metadata, identify columns that are entirely null, and inspect cooling and pricing attributes before selecting the final dimension columns.

In [31]:
from pyspark.sql import functions as F

# Load the 5-row Server Dimension
server_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/dim_89server.csv")
)

print("Rows:", server_df.count())
print("Columns:", len(server_df.columns))
print("Column names:", server_df.columns)

# Check columns that are completely NULL
display(
    server_df.select([
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
        for c in server_df.columns
    ])
)

# Inspect cooling types
display(
    server_df.select("server_id", "cooling_type")
    .distinct()
    .orderBy("server_id")
)

# Inspect pricing metadata
display(
    server_df.select(
        "server_id",
        "price_usd",
        "price_low_usd",
        "price_high_usd",
        "price_currency",
        "price_basis",
        "price_confidence",
        "priced_system"
    )
)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 32, Finished, Available, Finished, False)

Rows: 5
Columns: 23
Column names: ['server_id', 'server_name', 'server_type', 'gpu_family', 'gpu_count', 'gpu_memory_per_gpu_GB', 'gpu_power_limit_per_gpu_W', 'gpu_memory_type', 'cooling_type', 'cpu_specs', 'ram_gb', 'storage_tb', 'source_url', 'price_usd', 'price_low_usd', 'price_high_usd', 'price_currency', 'price_basis', 'price_confidence', 'priced_system', 'price_recorded_date', 'price_source', 'price_notes']


SynapseWidget(Synapse.DataFrame, bfe59b11-7004-4665-b63a-f7cd652b9887)

SynapseWidget(Synapse.DataFrame, 9f324312-5b35-499d-8a93-2b78c4cdcd03)

SynapseWidget(Synapse.DataFrame, 0da0707b-f591-4b3b-b46c-da83b7527f18)

### Clean the server dimension

Keep the attributes needed for server comparison and standardize the cooling-type labels while preserving the original CSV file.

In [34]:
from pyspark.sql import functions as F

# Keep the original CSV unchanged.
# Prepare the cleaned Server Dimension.

cooling_map = {
    "S001": "Liquid-Assisted",
    "S002": "Air Cooling",
    "S003": "High-Density",
    "S004": "High-Density",
    "S005": "Liquid Cooling"
}

keep_cols = [
    "server_id",
    "server_name",
    "server_type",
    "gpu_family",
    "gpu_count",
    "gpu_memory_per_gpu_GB",
    "gpu_power_limit_per_gpu_W",
    "gpu_memory_type",
    "cooling_type",
    "cpu_specs",
    "price_usd",
    "price_low_usd",
    "price_high_usd",
    "price_currency",
    "price_basis",
    "price_confidence",
    "priced_system"
]

dim_server_clean = server_df.select(*keep_cols)

# Shorten cooling descriptions using the verified server IDs
dim_server_clean = dim_server_clean.withColumn(
    "cooling_type",
    F.create_map(
        *[
            item
            for sid, label in cooling_map.items()
            for item in [F.lit(sid), F.lit(label)]
        ]
    )[F.col("server_id")]
)

# Validate the dimension before saving
assert dim_server_clean.count() == 5
assert dim_server_clean.select("server_id").distinct().count() == 5
assert dim_server_clean.filter(F.col("server_id").isNull()).count() == 0

print("Rows:", dim_server_clean.count())
print("Columns:", len(dim_server_clean.columns))

display(dim_server_clean.orderBy("server_id"))

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 35, Finished, Available, Finished, False)

Rows: 5
Columns: 17


SynapseWidget(Synapse.DataFrame, 6ce983d1-4e2f-4dd1-99d1-74aef4f3078c)

### Add the server surrogate key

Assign a stable key within this saved dimension build, ordered by `server_id`, so the fact table can reference each server.

In [35]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = Window.orderBy("server_id")

dim_server_final = dim_server_clean.withColumn(
    "server_key",
    F.row_number().over(window_spec)
)

dim_server_final = dim_server_final.select(
    "server_key",
    *dim_server_clean.columns
)

display(dim_server_final.orderBy("server_key"))

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff5cc358-3a52-419f-9f59-8486aa644452)

### Save the server dimension

Persist the prepared server dimension as a Delta table in the Lakehouse. The write stops if a table with this name already exists.

In [41]:
dim_server_final.write.format("delta").mode("errorifexists").saveAsTable("dim_server")

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 42, Finished, Available, Finished, False)

### Load and inspect the model dimension

Read the model metadata from the source CSV and inspect its schema and contents before adding a surrogate key.

In [43]:
model_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/dim_model.csv")
)

model_df.printSchema()
display(model_df)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 44, Finished, Available, Finished, False)

root
 |-- model_id: string (nullable = true)
 |-- workload_name: string (nullable = true)
 |-- model_size: integer (nullable = true)
 |-- image_size: integer (nullable = true)
 |-- batch_size: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, a0aedca2-daa9-452b-8787-21a9901d7135)

### Add the model surrogate key

Create `model_key` and place it first in the model dimension, retaining the original model attributes.

In [44]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.orderBy("model_id")

dim_model_final = model_df.withColumn(
    "model_key",
    F.row_number().over(window_spec)
)

# Put the surrogate key first
dim_model_final = dim_model_final.select(
    "model_key",
    *model_df.columns
)

display(dim_model_final)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8a092def-7357-405e-bd57-e26142f6f666)

### Save the model dimension

Persist the model dimension as a Delta table for use in the fact-table join.

In [45]:
dim_model_final.write.format("delta").mode("errorifexists").saveAsTable("dim_model")

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 46, Finished, Available, Finished, False)

### Build the date dimension

Extract distinct dates from the cleaned telemetry and derive a date key plus calendar attributes for time-based analysis.

In [47]:
from pyspark.sql import functions as F


telemetry_df = spark.table("processed_telemetry")


dim_date_final = (
    telemetry_df
    .select(F.to_date("date").alias("full_date"))
    .filter(F.col("full_date").isNotNull())
    .distinct()
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    .withColumn(
        "day_of_week_num",
        F.dayofweek("full_date")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin([1, 7])
    )
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_of_week_num",
        "is_weekend"
    )
    .orderBy("date_key")
)

display(dim_date_final)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7306559a-75ae-4dba-bcd8-8b9f83098a54)

### Validate the date dimension

Compare the number of dates with the number of distinct date keys and inspect the resulting schema before saving.

In [48]:
print("Date count:", dim_date_final.count())
print("Distinct date keys:", dim_date_final.select("date_key").distinct().count())

dim_date_final.printSchema()

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 49, Finished, Available, Finished, False)

Date count: 278
Distinct date keys: 278
root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- day_of_week_num: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)



### Save the date dimension

Persist the date dimension as a Delta table so each telemetry reading can reference its calendar date.

In [49]:
dim_date_final.write.format("delta").mode("errorifexists").saveAsTable("dim_date")

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 50, Finished, Available, Finished, False)

### Build the fact-table staging DataFrame

Use broadcast joins with the small server and model dimensions and join the date dimension to add all three surrogate keys while retaining every source reading.

In [51]:
from pyspark.sql import functions as F


telemetry_df = spark.table("processed_telemetry")

dim_server_lookup = (
    spark.table("dim_server")
    .select("server_id", "server_key")
)

dim_model_lookup = (
    spark.table("dim_model")
    .select("model_id", "model_key")
)

dim_date_lookup = (
    spark.table("dim_date")
    .select(
        F.col("full_date").alias("date_lookup"),
        "date_key"
    )
)


fact_stage = (
    telemetry_df
    .join(
        F.broadcast(dim_server_lookup),
        on="server_id",
        how="left"
    )
    .join(
        F.broadcast(dim_model_lookup),
        on="model_id",
        how="left"
    )
    .join(
        F.broadcast(dim_date_lookup),
        F.to_date(telemetry_df["date"]) == F.col("date_lookup"),
        how="left"
    )
    .drop("date_lookup")
    .withColumnRenamed("timestamp", "reading_ts")
)

print("Source rows:", telemetry_df.count())
print("Fact stage rows:", fact_stage.count())

display(fact_stage.limit(5))

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 52, Finished, Available, Finished, False)

Source rows: 2000000
Fact stage rows: 2000000


SynapseWidget(Synapse.DataFrame, 294bf3b4-b327-41eb-bcfc-67c50fe80600)

### Check the staged foreign keys

Verify the row count and confirm that the server, model, and date keys were successfully matched before finalizing the fact table.

In [52]:
# Check for missing dimension keys
fact_stage.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.when(F.col("server_key").isNull(), 1).otherwise(0)
    ).alias("missing_server_key"),
    F.sum(
        F.when(F.col("model_key").isNull(), 1).otherwise(0)
    ).alias("missing_model_key"),
    F.sum(
        F.when(F.col("date_key").isNull(), 1).otherwise(0)
    ).alias("missing_date_key")
).show(truncate=False)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 53, Finished, Available, Finished, False)

+----------+------------------+-----------------+----------------+
|total_rows|missing_server_key|missing_model_key|missing_date_key|
+----------+------------------+-----------------+----------------+
|2000000   |0                 |0                |0               |
+----------+------------------+-----------------+----------------+



### Inspect the staged fact schema

Review the staged row count, column list, schema, and sample key values before selecting the final fact columns.

In [53]:
print("Rows:", fact_stage.count())
print("Columns:", len(fact_stage.columns))

fact_stage.printSchema()

fact_stage.select(
    "server_key",
    "model_key",
    "date_key",
    "reading_ts",
    "experiment_id"
).show(5, truncate=False)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 54, Finished, Available, Finished, False)

Rows: 2000000
Columns: 30
root
 |-- model_id: string (nullable = true)
 |-- server_id: string (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- time: string (nullable = true)
 |-- experiment_id: string (nullable = true)
 |-- gpu_count: integer (nullable = true)
 |-- gpu_utilization_avg_pct: double (nullable = true)
 |-- gpu_power_total_W: double (nullable = true)
 |-- gpu_temp_max_C: double (nullable = true)
 |-- cpu_utilization_percent: double (nullable = true)
 |-- cpu_freq_MHz: double (nullable = true)
 |-- cpu_power_W: double (nullable = true)
 |-- cpu_temp_C: double (nullable = true)
 |-- ambient_temp_C: double (nullable = true)
 |-- inlet_temp_C: double (nullable = true)
 |-- exhaust_temp_C: double (nullable = true)
 |-- humidity_pct: double (nullable = true)
 |-- cooling_type: string (nullable = true)
 |-- cooling_power_W: double (nullable = true)
 |-- fan_speed_RPM: double (nullable = true)
 |-- airflow_CFM: double (nullable =

### Select the final fact-table columns

Keep the three dimension keys, reading timestamp, source identifiers, experiment ID, and telemetry measures; omit redundant date/time columns and avoid duplicate `reading_ts`.

In [56]:
key_cols = [
    "server_key",
    "model_key",
    "date_key"
]

source_id_cols = [
    "server_id",
    "model_id",
    "experiment_id"
]

excluded_cols = {
    "date",
    "time",
    "timestamp",
    "server_key",
    "model_key",
    "date_key",
    "reading_ts"
}

measure_cols = [
    c for c in fact_stage.columns
    if c not in excluded_cols
    and c not in source_id_cols
]

fact_telemetry_final = fact_stage.select(
    *key_cols,
    "reading_ts",
    *source_id_cols,
    *measure_cols
)

print("Rows:", fact_telemetry_final.count())
print("Columns:", len(fact_telemetry_final.columns))

fact_telemetry_final.printSchema()
display(fact_telemetry_final.limit(5))

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 57, Finished, Available, Finished, False)

Rows: 2000000
Columns: 28
root
 |-- server_key: integer (nullable = true)
 |-- model_key: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- server_id: string (nullable = true)
 |-- model_id: string (nullable = true)
 |-- experiment_id: string (nullable = true)
 |-- gpu_count: integer (nullable = true)
 |-- gpu_utilization_avg_pct: double (nullable = true)
 |-- gpu_power_total_W: double (nullable = true)
 |-- gpu_temp_max_C: double (nullable = true)
 |-- cpu_utilization_percent: double (nullable = true)
 |-- cpu_freq_MHz: double (nullable = true)
 |-- cpu_power_W: double (nullable = true)
 |-- cpu_temp_C: double (nullable = true)
 |-- ambient_temp_C: double (nullable = true)
 |-- inlet_temp_C: double (nullable = true)
 |-- exhaust_temp_C: double (nullable = true)
 |-- humidity_pct: double (nullable = true)
 |-- cooling_type: string (nullable = true)
 |-- cooling_power_W: double (nullable = true)
 |-- fan_speed_RPM: doubl

SynapseWidget(Synapse.DataFrame, 4ee5e1dd-53b2-4974-bbcc-ad057c0949f5)

### Save the fact table

Persist the finalized telemetry fact as a Delta table. The write stops if `fact_telemetry` already exists.

In [57]:
fact_telemetry_final.write \
    .format("delta") \
    .mode("errorifexists") \
    .saveAsTable("fact_telemetry")

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 58, Finished, Available, Finished, False)

### Verify the saved fact table

Reload the saved table and confirm its row count, column count, and schema.

In [58]:
fact_check = spark.table("fact_telemetry")

print("Rows:", fact_check.count())
print("Columns:", len(fact_check.columns))

fact_check.printSchema()

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 59, Finished, Available, Finished, False)

Rows: 2000000
Columns: 28
root
 |-- server_key: integer (nullable = true)
 |-- model_key: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- server_id: string (nullable = true)
 |-- model_id: string (nullable = true)
 |-- experiment_id: string (nullable = true)
 |-- gpu_count: integer (nullable = true)
 |-- gpu_utilization_avg_pct: double (nullable = true)
 |-- gpu_power_total_W: double (nullable = true)
 |-- gpu_temp_max_C: double (nullable = true)
 |-- cpu_utilization_percent: double (nullable = true)
 |-- cpu_freq_MHz: double (nullable = true)
 |-- cpu_power_W: double (nullable = true)
 |-- cpu_temp_C: double (nullable = true)
 |-- ambient_temp_C: double (nullable = true)
 |-- inlet_temp_C: double (nullable = true)
 |-- exhaust_temp_C: double (nullable = true)
 |-- humidity_pct: double (nullable = true)
 |-- cooling_type: string (nullable = true)
 |-- cooling_power_W: double (nullable = true)
 |-- fan_speed_RPM: doubl

### Validate foreign-key completeness

Check that all fact rows contain non-null server, model, and date keys.

In [59]:
from pyspark.sql import functions as F

fact = spark.table("fact_telemetry")

print("Fact rows:", fact.count())

print("Missing foreign keys:")
fact.select(
    F.sum(F.when(F.col("server_key").isNull(), 1).otherwise(0))
        .alias("missing_server_key"),
    F.sum(F.when(F.col("model_key").isNull(), 1).otherwise(0))
        .alias("missing_model_key"),
    F.sum(F.when(F.col("date_key").isNull(), 1).otherwise(0))
        .alias("missing_date_key")
).show()

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 60, Finished, Available, Finished, False)

Fact rows: 2000000
Missing foreign keys:
+------------------+-----------------+----------------+
|missing_server_key|missing_model_key|missing_date_key|
+------------------+-----------------+----------------+
|                 0|                0|               0|
+------------------+-----------------+----------------+



### Validate the fact grain

Check for duplicate readings using the agreed natural key: reading timestamp, server ID, and experiment ID.

In [60]:
from pyspark.sql import functions as F

fact = spark.table("fact_telemetry")

duplicate_groups = (
    fact.groupBy(
        "reading_ts",
        "server_id",
        "experiment_id"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate key groups:", duplicate_groups.count())

duplicate_groups.show(10, truncate=False)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 61, Finished, Available, Finished, False)

Duplicate key groups: 0
+----------+---------+-------------+-----+
|reading_ts|server_id|experiment_id|count|
+----------+---------+-------------+-----+
+----------+---------+-------------+-----+



### Validate core telemetry measures

Check utilization ranges, negative power values, and whether GPU memory usage exceeds total memory.

In [61]:
from pyspark.sql import functions as F

fact = spark.table("fact_telemetry")

checks = fact.select(
    F.sum(
        F.when(
            (F.col("gpu_utilization_avg_pct") < 0) |
            (F.col("gpu_utilization_avg_pct") > 100), 1
        ).otherwise(0)
    ).alias("invalid_gpu_utilization"),

    F.sum(
        F.when(
            (F.col("cpu_utilization_percent") < 0) |
            (F.col("cpu_utilization_percent") > 100), 1
        ).otherwise(0)
    ).alias("invalid_cpu_utilization"),

    F.sum(
        F.when(F.col("gpu_power_total_W") < 0, 1).otherwise(0)
    ).alias("negative_gpu_power"),

    F.sum(
        F.when(F.col("cpu_power_W") < 0, 1).otherwise(0)
    ).alias("negative_cpu_power"),

    F.sum(
        F.when(
            F.col("gpu_memory_used_total_GB") >
            F.col("gpu_memory_total_GB"), 1
        ).otherwise(0)
    ).alias("memory_used_exceeds_total")
)

checks.show(truncate=False)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 62, Finished, Available, Finished, False)

+-----------------------+-----------------------+------------------+------------------+-------------------------+
|invalid_gpu_utilization|invalid_cpu_utilization|negative_gpu_power|negative_cpu_power|memory_used_exceeds_total|
+-----------------------+-----------------------+------------------+------------------+-------------------------+
|0                      |0                      |0                 |0                 |0                        |
+-----------------------+-----------------------+------------------+------------------+-------------------------+



### Validate dimension-key integrity

Check that each dimension's surrogate keys are unique and that every distinct foreign key in the fact matches a dimension row.

In [62]:
from pyspark.sql import functions as F

fact = spark.table("fact_telemetry")
server_dim = spark.table("dim_server")
model_dim = spark.table("dim_model")
date_dim = spark.table("dim_date")

print("Duplicate server keys:",
      server_dim.groupBy("server_key").count()
      .filter(F.col("count") > 1).count())

print("Duplicate model keys:",
      model_dim.groupBy("model_key").count()
      .filter(F.col("count") > 1).count())

print("Duplicate date keys:",
      date_dim.groupBy("date_key").count()
      .filter(F.col("count") > 1).count())

print("Unmatched server keys:",
      fact.select("server_key").distinct()
      .join(server_dim.select("server_key"),
            on="server_key", how="left_anti").count())

print("Unmatched model keys:",
      fact.select("model_key").distinct()
      .join(model_dim.select("model_key"),
            on="model_key", how="left_anti").count())

print("Unmatched date keys:",
      fact.select("date_key").distinct()
      .join(date_dim.select("date_key"),
            on="date_key", how="left_anti").count())

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 63, Finished, Available, Finished, False)

Duplicate server keys: 0
Duplicate model keys: 0
Duplicate date keys: 0
Unmatched server keys: 0
Unmatched model keys: 0
Unmatched date keys: 0


### Review Lakehouse tables

List the tables available after the warehouse and validation steps, before running the business-analysis queries.

In [67]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 68, Finished, Available, Finished, False)

+---------+-------------------+-----------+
|namespace|tableName          |isTemporary|
+---------+-------------------+-----------+
|mylake   |processed_telemetry|false      |
|mylake   |dim_server         |false      |
|mylake   |dim_model          |false      |
|mylake   |dim_date           |false      |
|mylake   |fact_telemetry     |false      |
+---------+-------------------+-----------+



Business Queries using Spark SQL

In [68]:
%%sql
-- Compares server performance, resource utilization, and hardware cost
SELECT
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.gpu_count,
    s.gpu_memory_per_gpu_GB,
    s.price_usd,

    COUNT(*) AS reading_count,

    ROUND(AVG(f.gpu_utilization_avg_pct), 2)
        AS avg_gpu_utilization_pct,

    ROUND(
        percentile_approx(f.gpu_utilization_avg_pct, 0.95),
        2
    ) AS p95_gpu_utilization_pct,

    ROUND(
        AVG(
            CASE
                WHEN f.gpu_memory_total_GB > 0
                THEN f.gpu_memory_used_total_GB
                     / f.gpu_memory_total_GB * 100
            END
        ),
        2
    ) AS avg_gpu_memory_utilization_pct,

    ROUND(AVG(f.gpu_temp_max_C), 2)
        AS avg_gpu_temp_C,

    ROUND(MAX(f.gpu_temp_max_C), 2)
        AS peak_gpu_temp_C,

    ROUND(AVG(f.gpu_power_total_W), 2)
        AS avg_gpu_power_W,

    ROUND(AVG(f.cpu_utilization_percent), 2)
        AS avg_cpu_utilization_pct

FROM mylake.fact_telemetry f

JOIN mylake.dim_server s
    ON f.server_key = s.server_key

GROUP BY
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.gpu_count,
    s.gpu_memory_per_gpu_GB,
    s.price_usd

ORDER BY avg_gpu_utilization_pct DESC

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 69, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 14 fields>

In [69]:
%%sql
-- Compares GPU power consumption with GPU utilization
-- to identify servers with lower power demand under the observed workload.

SELECT
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.price_usd,

    ROUND(AVG(f.gpu_utilization_avg_pct), 2)
        AS avg_gpu_utilization_pct,

    ROUND(AVG(f.gpu_power_total_W), 2)
        AS avg_gpu_power_W,

    ROUND(MAX(f.gpu_power_total_W), 2)
        AS peak_gpu_power_W,

    ROUND(
        AVG(f.gpu_power_total_W)
        / NULLIF(AVG(f.gpu_utilization_avg_pct), 0),
        2
    ) AS avg_power_per_utilization_point

FROM mylake.fact_telemetry f

JOIN mylake.dim_server s
    ON f.server_key = s.server_key

GROUP BY
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.price_usd

ORDER BY avg_gpu_power_W ASC

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 70, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 8 fields>

In [70]:
%%sql
-- Compares average and peak GPU temperatures across servers
-- to assess thermal behavior under the observed AI workload.

SELECT
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.cooling_type,

    ROUND(AVG(f.gpu_temp_max_C), 2)
        AS avg_gpu_temp_C,

    ROUND(MAX(f.gpu_temp_max_C), 2)
        AS peak_gpu_temp_C,

    ROUND(AVG(f.gpu_utilization_avg_pct), 2)
        AS avg_gpu_utilization_pct,

    ROUND(AVG(f.gpu_power_total_W), 2)
        AS avg_gpu_power_W

FROM mylake.fact_telemetry f

JOIN mylake.dim_server s
    ON f.server_key = s.server_key

GROUP BY
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.cooling_type

ORDER BY avg_gpu_temp_C ASC

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 71, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 8 fields>

In [71]:
%%sql
-- Compares GPU utilization against power consumption
-- to identify servers with better utilization per kW.

SELECT
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.gpu_count,
    s.price_usd,

    ROUND(AVG(f.gpu_utilization_avg_pct), 2)
        AS avg_gpu_utilization_pct,

    ROUND(AVG(f.gpu_power_total_W), 2)
        AS avg_gpu_power_W,

    ROUND(
        AVG(f.gpu_utilization_avg_pct)
        / NULLIF(AVG(f.gpu_power_total_W) / 1000.0, 0),
        2
    ) AS gpu_utilization_per_kW

FROM mylake.fact_telemetry f

JOIN mylake.dim_server s
    ON f.server_key = s.server_key

GROUP BY
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.gpu_count,
    s.price_usd

ORDER BY gpu_utilization_per_kW DESC

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 72, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 8 fields>

In [72]:
%%sql
-- Compares estimated server purchase price with observed GPU utilization
-- and power efficiency to support cost-aware server selection.

SELECT
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.price_usd,

    ROUND(AVG(f.gpu_utilization_avg_pct), 2)
        AS avg_gpu_utilization_pct,

    ROUND(AVG(f.gpu_power_total_W), 2)
        AS avg_gpu_power_W,

    ROUND(
        AVG(f.gpu_utilization_avg_pct)
        / NULLIF(AVG(f.gpu_power_total_W) / 1000.0, 0),
        2
    ) AS gpu_utilization_per_kW

FROM mylake.fact_telemetry f

JOIN mylake.dim_server s
    ON f.server_key = s.server_key

GROUP BY
    s.server_id,
    s.server_name,
    s.gpu_family,
    s.price_usd

ORDER BY s.price_usd ASC

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 73, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 7 fields>

In [74]:
%%sql
-- Analyzes hourly server performance to identify potential temperature,
-- power-consumption, and GPU-utilization spikes over time.

SELECT
    s.server_id,
    s.gpu_family,
    DATE_TRUNC('hour', f.Reading_ts) AS hour_start,

    ROUND(AVG(f.gpu_utilization_avg_pct), 2)
        AS avg_gpu_utilization_pct,

    ROUND(AVG(f.gpu_power_total_W), 2)
        AS avg_gpu_power_W,

    ROUND(MAX(f.gpu_power_total_W), 2)
        AS peak_gpu_power_W,

    ROUND(AVG(f.gpu_temp_max_C), 2)
        AS avg_gpu_temp_C,

    ROUND(MAX(f.gpu_temp_max_C), 2)
        AS peak_gpu_temp_C,

    COUNT(*) AS reading_count

FROM mylake.fact_telemetry f

JOIN mylake.dim_server s
    ON f.server_key = s.server_key

GROUP BY
    s.server_id,
    s.gpu_family,
    DATE_TRUNC('hour', f.Reading_ts)

ORDER BY
    hour_start,
    s.server_id;

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 75, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 9 fields>

In [75]:
%%sql
-- Compares server cooling types with GPU thermal performance,
-- utilization, and power consumption.

SELECT
    s.server_id,
    s.gpu_family,
    s.cooling_type,
    s.price_usd,

    ROUND(AVG(f.gpu_utilization_avg_pct), 2)
        AS avg_gpu_utilization_pct,

    ROUND(AVG(f.gpu_temp_max_C), 2)
        AS avg_gpu_temp_C,

    ROUND(MAX(f.gpu_temp_max_C), 2)
        AS peak_gpu_temp_C,

    ROUND(AVG(f.gpu_power_total_W), 2)
        AS avg_gpu_power_W

FROM mylake.fact_telemetry f

JOIN mylake.dim_server s
    ON f.server_key = s.server_key

GROUP BY
    s.server_id,
    s.gpu_family,
    s.cooling_type,
    s.price_usd

ORDER BY avg_gpu_temp_C DESC;

StatementMeta(, ab2bf6e7-0599-487f-906f-b5d5e02f64ed, 76, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 8 fields>